# Box Point Path Planner

Plane eine lokale Trajektorie durch alle Target-Punkte einer Box (Start = erste Pose in der Box, Ziel = erste Pose der nächsten Box) und visualisiere die resultierenden SE3-Posen.

In [40]:
import json
import numpy as np
import dill
from pathlib import Path
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from scipy.spatial.transform import Rotation as R
import sys

NOTEBOOK_DIR = Path().resolve()
MODULE_ROOT = (NOTEBOOK_DIR / '../..').resolve()
if str(MODULE_ROOT) not in sys.path:
    sys.path.append(str(MODULE_ROOT))

import mathUtils

trajectory_path = Path("gt_coveragewithTargets_withBoxes.dill")  # Trajektorie mit Box- und TargetPoint-Metadaten


In [41]:
# Tool transform (base -> tool)
tool_translation = np.array([1.5, 0.0, 0.0])  # [m] in base frame
tool_rotation_euler_deg = (0.0, 0.0, 0.0)  # roll, pitch, yaw in degrees

tool_rotation = R.from_euler('xyz', tool_rotation_euler_deg, degrees=True).as_matrix()
rotation_weight = 0.5  # Gewichtung der Drehkosten ("Meter" pro Rad)


## Tool-aware PFADPLANUNG

- Die Trajektorie berücksichtigt ein Werkzeug mit frei einstellbarer SE(3)-Pose relativ zum Roboter-Basiskoordinatensystem (`tool_translation`, `tool_rotation`).
- Nur die Werkzeugspitze muss die `TargetPoints` erreichen. Dazu planen wir zunächst im Werkzeugraum (Start-/Zielpose + Zielpunkte).
- Für bis zu 8 Punkte berechnen wir die optimale Besuchsreihenfolge exakt (Permutation mit minimaler Länge). Darüber nutzen wir eine zielbewusste Greedy-Heuristik.
- Anschließend rekonstruieren wir die Roboter-Basiskonfigurationen, indem wir die Werkzeugtrajektorie mit dem inversen Werkzeug-Offset kombinieren. Dadurch treffen wir garantiert die nächste Box-Startpose.


### Mathematische Beschreibung der Besuchsreihenfolge

Seien $s$ die Werkzeugposition an der ersten Pose der aktuellen Box und $g$ die Werkzeugposition an der ersten Pose der Folgebox. Die Menge der Zielpunkte innerhalb der Box ist $P = \{p_1, \dots, p_n\}$.

Wir minimieren die Länge
\[
  L(\pi) = \|s - p_{\pi_1}\| + \sum_{k=1}^{n-1} \|p_{\pi_k} - p_{\pi_{k+1}}\| + \|p_{\pi_n} - g\| + \lambda \sum_{k=0}^{n} |\Delta \psi_k|,
\]
wobei $\lambda = 	exttt{rotation\_weight}$, $\Delta \psi_k$ die jeweilige Differenz zwischen den Yaw-Winkeln aufeinanderfolgender Segmente ist und $\psi_0$ bzw. $\psi_{n+1}$ die Start- und Zielausrichtung darstellen.

- Für $n \leq 8$ enumerieren wir alle $n!$ Permutationen und wählen diejenige mit minimalem $L(\pi)$ (exakte Lösung).
- Für größere $n$ nutzen wir eine Greedy-Heuristik: Wir wählen iterativ den Punkt, der $\|p_j-c\| + \lambda |\psi_j - \psi_c| + 0{,}2\,(\|p_j-g\| + \lambda |\psi_g - \psi_j|)$ minimiert (aktueller Werkzeugpunkt $c$ mit Yaw $\psi_c$).
- Nachdem die Reihenfolge steht, rekonstruiert der Planner die Basiskonfigurationen, indem der Werkzeugoffset entfernt wird; dadurch treffen wir Start- und Zielpose der benachbarten Boxen.


In [42]:
with open(trajectory_path, "rb") as f:
    trajectory = dill.load(f)

poses = np.array([pose[:3, 3] for pose in trajectory.poses_se3])
rotations = np.array([pose[:3, :3] for pose in trajectory.poses_se3])
boxes = trajectory.meta.get("TargetBoxes", [])
target_points = np.asarray(trajectory.meta.get("TargetPoints", []))

print(f"Posen: {len(poses)}  |  Boxen: {len(boxes)}  |  TargetPoints: {len(target_points)}")
if len(boxes) < 2:
    raise ValueError("Mindestens zwei Boxen erforderlich, um eine Verbindung zu planen.")


Posen: 2741  |  Boxen: 119  |  TargetPoints: 600


In [43]:
def oriented_basis(box):
    direction = np.array(box["direction"], dtype=float)
    direction[2] = 0.0
    dir_norm = np.linalg.norm(direction[:2])
    if dir_norm < 1e-8:
        direction = np.array([1.0, 0.0, 0.0])
    else:
        direction /= dir_norm
    lateral = np.array([-direction[1], direction[0], 0.0])
    return direction, lateral


def filter_points_in_box(points, box):
    if points is None or len(points) == 0:
        return np.empty((0, 3))
    center = np.array(box["center"], dtype=float)
    length = float(box["length"])
    width = float(box["width"])
    direction, lateral = oriented_basis(box)
    rel = np.asarray(points) - center
    proj_forward = rel @ direction
    proj_side = rel @ lateral
    mask = (np.abs(proj_forward) <= length / 2.0) & (np.abs(proj_side) <= width / 2.0)
    selected = np.asarray(points)[mask]
    return selected


def first_pose_index_in_box(poses, box):
    if poses is None or len(poses) == 0:
        return None, None
    center = np.array(box["center"], dtype=float)
    length = float(box["length"])
    width = float(box["width"])
    direction, lateral = oriented_basis(box)
    rel = poses - center
    proj_forward = rel @ direction
    proj_side = rel @ lateral
    mask = (np.abs(proj_forward) <= length / 2.0) & (np.abs(proj_side) <= width / 2.0)
    idx_candidates = np.where(mask)[0]
    if len(idx_candidates) == 0:
        return None, None
    idx = int(idx_candidates[0])
    return idx, poses[idx], rotations[idx]


In [44]:
from itertools import permutations

MAX_REVERSE_DISTANCE = 2.0
REVERSE_TRANSLATION_PENALTY = 0.3
LATERAL_REVERSE_TOL = 0.5


def angle_wrap(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi


def yaw_from_vector(vec, fallback):
    if np.linalg.norm(vec[:2]) < 1e-9:
        return fallback
    return float(np.arctan2(vec[1], vec[0]))


def compute_base_transition(tool_point, base_pos, base_yaw, allow_reverse=False):
    base_pos = np.asarray(base_pos, dtype=float)
    current_rot = R.from_euler('z', base_yaw).as_matrix()
    current_tool = base_pos + current_rot @ tool_translation
    direction = tool_point - current_tool
    forward_yaw = yaw_from_vector(direction, base_yaw)
    forward_rot = R.from_euler('z', forward_yaw).as_matrix()
    forward_base = tool_point - forward_rot @ tool_translation
    forward_translation = np.linalg.norm(forward_base - base_pos)
    forward_rot_change = abs(angle_wrap(forward_yaw - base_yaw))
    best = (forward_base, forward_yaw, forward_translation, forward_rot_change)
    if allow_reverse:
        body_dir = current_rot.T @ direction
        lateral_offset = abs(body_dir[1])
        longitudinal = -body_dir[0]
        if longitudinal > 0 and longitudinal <= MAX_REVERSE_DISTANCE and lateral_offset <= LATERAL_REVERSE_TOL:
            backward_base = base_pos + direction
            backward_translation = longitudinal
            forward_cost = forward_translation + rotation_weight * forward_rot_change
            backward_cost = backward_translation * (1.0 + REVERSE_TRANSLATION_PENALTY)
            if backward_cost < forward_cost:
                best = (backward_base, base_yaw, backward_translation, 0.0)
    return best


def path_length_distance(order, points, start_base, start_yaw, goal_base, allow_reverse=False):
    if len(order) == 0:
        return float(np.linalg.norm(goal_base - start_base))
    total = 0.0
    current_base = start_base.copy()
    current_yaw = start_yaw
    for idx in order:
        current_base, current_yaw, translation, _ = compute_base_transition(points[idx], current_base, current_yaw, allow_reverse=allow_reverse)
        total += translation
    total += np.linalg.norm(goal_base - current_base)
    return float(total)


def best_visit_order_distance(points, start_base, start_yaw, goal_base, brute_force_limit=8, allow_reverse=False):
    points = np.asarray(points)
    n = len(points)
    if n == 0:
        return []
    indices = list(range(n))
    if 0 < n <= brute_force_limit:
        best = min(
            permutations(indices),
            key=lambda order: path_length_distance(order, points, start_base, start_yaw, goal_base, allow_reverse=allow_reverse),
        )
        return list(best)
    visited = []
    remaining = indices.copy()
    current_base = start_base.copy()
    current_yaw = start_yaw
    while remaining:
        def score(idx):
            candidate_base, _, translation, _ = compute_base_transition(points[idx], current_base, current_yaw, allow_reverse=allow_reverse)
            goal_cost = np.linalg.norm(goal_base - candidate_base)
            return translation + 0.2 * goal_cost
        best_idx = min(remaining, key=score)
        current_base, current_yaw, _ = compute_base_transition(points[best_idx], current_base, current_yaw, allow_reverse=allow_reverse)
        visited.append(best_idx)
        remaining.remove(best_idx)
    return visited


def path_length_rotation(order, points, start_base, start_yaw, goal_base, goal_yaw, allow_reverse=False):
    if len(order) == 0:
        translation = np.linalg.norm(goal_base - start_base)
        return translation + rotation_weight * abs(angle_wrap(goal_yaw - start_yaw))
    total = 0.0
    current_base = start_base.copy()
    current_yaw = start_yaw
    for idx in order:
        current_base, current_yaw, translation, rot_change = compute_base_transition(points[idx], current_base, current_yaw, allow_reverse=allow_reverse)
        total += translation
        if allow_reverse:
            total += rotation_weight * rot_change
    total += np.linalg.norm(goal_base - current_base)
    total += rotation_weight * abs(angle_wrap(goal_yaw - current_yaw))
    return float(total)


def best_visit_order_rotation(points, start_base, start_yaw, goal_base, goal_yaw, brute_force_limit=8, allow_reverse=False):
    points = np.asarray(points)
    n = len(points)
    if n == 0:
        return []
    indices = list(range(n))
    if 0 < n <= brute_force_limit:
        best = min(
            permutations(indices),
            key=lambda order: path_length_rotation(order, points, start_base, start_yaw, goal_base, goal_yaw, allow_reverse=allow_reverse),
        )
        return list(best)
    visited = []
    remaining = indices.copy()
    current_base = start_base.copy()
    current_yaw = start_yaw
    while remaining:
        def score(idx):
            candidate_base, candidate_yaw, translation, rot_change = compute_base_transition(points[idx], current_base, current_yaw, allow_reverse=allow_reverse)
            goal_cost = np.linalg.norm(goal_base - candidate_base)
            goal_rot = rotation_weight * abs(angle_wrap(goal_yaw - candidate_yaw))
            rot_change_scaled = rotation_weight * rot_change
            return translation + rot_change_scaled + 0.2 * (goal_cost + goal_rot)
        best_idx = min(remaining, key=score)
        current_base, current_yaw, _, _ = compute_base_transition(points[best_idx], current_base, current_yaw, allow_reverse=allow_reverse)
        visited.append(best_idx)
        remaining.remove(best_idx)
    return visited


def build_path_from_order(visit_order, start_rot, start_base, goal_rot, goal_base, points, allow_reverse=False):
    path = []
    path.append(build_pose(start_rot, start_base))
    current_base = start_base.copy()
    current_yaw = float(np.arctan2(start_rot[1, 0], start_rot[0, 0]))
    for idx in visit_order:
        base_pos, current_yaw, _, _ = compute_base_transition(points[idx], current_base, current_yaw, allow_reverse=allow_reverse)
        base_rot = R.from_euler('z', current_yaw).as_matrix()
        path.append(build_pose(base_rot, base_pos))
        current_base = base_pos
    path.append(build_pose(goal_rot, goal_base))
    return path


def plan_path_through_points_distance(start_pose, start_rot, intermediate_points, goal_pose, goal_rot):
    intermediate_points = np.asarray(intermediate_points)
    start_base = np.array(start_pose, dtype=float)
    goal_base = np.array(goal_pose, dtype=float)
    start_yaw = float(np.arctan2(start_rot[1, 0], start_rot[0, 0]))
    visit_order = best_visit_order_distance(intermediate_points, start_base, start_yaw, goal_base)
    path = build_path_from_order(visit_order, start_rot, start_base, goal_rot, goal_base, intermediate_points)
    return path, visit_order


def plan_path_through_points_rotation(start_pose, start_rot, intermediate_points, goal_pose, goal_rot):
    intermediate_points = np.asarray(intermediate_points)
    start_base = np.array(start_pose, dtype=float)
    goal_base = np.array(goal_pose, dtype=float)
    start_yaw = float(np.arctan2(start_rot[1, 0], start_rot[0, 0]))
    goal_yaw = float(np.arctan2(goal_rot[1, 0], goal_rot[0, 0]))
    visit_order = best_visit_order_rotation(intermediate_points, start_base, start_yaw, goal_base, goal_yaw)
    path = build_path_from_order(visit_order, start_rot, start_base, goal_rot, goal_base, intermediate_points)
    return path, visit_order


def plan_path_through_points_reverse(start_pose, start_rot, intermediate_points, goal_pose, goal_rot):
    intermediate_points = np.asarray(intermediate_points)
    start_base = np.array(start_pose, dtype=float)
    goal_base = np.array(goal_pose, dtype=float)
    start_yaw = float(np.arctan2(start_rot[1, 0], start_rot[0, 0]))
    goal_yaw = float(np.arctan2(goal_rot[1, 0], goal_rot[0, 0]))
    visit_order = best_visit_order_rotation(
        intermediate_points,
        start_base,
        start_yaw,
        goal_base,
        goal_yaw,
        allow_reverse=True,
    )
    path = build_path_from_order(visit_order, start_rot, start_base, goal_rot, goal_base, intermediate_points, allow_reverse=True)
    return path, visit_order


In [45]:
def build_pose(rotation, translation):
    T = np.eye(4)
    T[:3, :3] = rotation
    T[:3, 3] = translation
    return T


In [46]:
def poses_to_generator_points(poses, default_velocity=1.0):
    pts = []
    for pose in poses:
        translation = pose[:3, 3]
        euler = R.from_matrix(pose[:3, :3]).as_euler('zyx', degrees=True)
        yaw_deg, pitch_deg, roll_deg = euler
        pts.append((translation[0], translation[1], translation[2], yaw_deg, roll_deg, pitch_deg, default_velocity))
    return pts


def smooth_plan(poses, sampling_rate=20):
    if poses is None or len(poses) < 2:
        return []
    points = poses_to_generator_points(poses)
    traj = mathUtils.trajectory_generation(points, use_yaw_input=False, use_pitch_input=False, sampling_rate=sampling_rate)
    if traj is None:
        return []
    return list(traj.poses_se3)


def add_pose_axis_markers(fig, poses, legendgroup, scale=0.35, max_markers=25):
    if not poses:
        return
    step = max(1, len(poses) // max_markers)
    for pose in poses[::step]:
        draw_pose_axes(fig, pose, scale=scale, opacity=0.3, legendgroup=legendgroup)


In [47]:
def draw_pose_axes(fig, pose_matrix, scale=0.7, opacity=0.6, legendgroup=None):
    origin = pose_matrix[:3, 3]
    axes = pose_matrix[:3, :3]
    colors = [("Pose X", "red"), ("Pose Y", "green"), ("Pose Z", "blue")]
    for i, (label, color) in enumerate(colors):
        axis_vec = axes[:, i]
        end = origin + axis_vec * scale
        fig.add_trace(
            go.Scatter3d(
                x=[origin[0], end[0]],
                y=[origin[1], end[1]],
                z=[origin[2], end[2]],
                mode="lines",
                name=label,
                line=dict(color=color, width=3),
                opacity=opacity,
                showlegend=False,
                legendgroup=legendgroup,
            )
        )


In [48]:
def build_visit_annotation(points, visit_order, prefix=""):
    if not len(visit_order):
        return None, None
    ordered_points = np.array([points[idx] for idx in visit_order])
    labels = [f"{prefix}{i + 1}" for i in range(len(visit_order))]
    return ordered_points, labels


In [49]:
def visualize_plan(box_idx, zoom_box=False, smooth_enabled=True):
    current_box = boxes[box_idx]
    next_box = boxes[box_idx + 1]
    pose_idx, start_pose, start_rot = first_pose_index_in_box(poses, current_box)
    next_pose_idx, goal_pose, goal_rot = first_pose_index_in_box(poses, next_box)
    if pose_idx is None or next_pose_idx is None:
        raise ValueError("Konnte Start/Ziel-Pose nicht bestimmen")
    points = filter_points_in_box(target_points, current_box)
    planned_path_dist, visit_order_dist = plan_path_through_points_distance(start_pose, start_rot, points, goal_pose, goal_rot)
    planned_path_rot, visit_order_rot = plan_path_through_points_rotation(start_pose, start_rot, points, goal_pose, goal_rot)
    planned_path_rev, visit_order_rev = plan_path_through_points_reverse(start_pose, start_rot, points, goal_pose, goal_rot)
    smooth_dist = smooth_plan(planned_path_dist) if smooth_enabled else []
    smooth_rot = smooth_plan(planned_path_rot) if smooth_enabled else []
    smooth_rev = smooth_plan(planned_path_rev) if smooth_enabled else []

    fig = go.Figure()
    fig.add_trace(
        go.Scatter3d(
            x=poses[:, 0],
            y=poses[:, 1],
            z=poses[:, 2],
            mode="lines",
            name="Trajektorie",
            line=dict(color="steelblue", width=3),
            opacity=0.25,
        )
    )
    if len(points):
        fig.add_trace(
            go.Scatter3d(
                x=points[:, 0],
                y=points[:, 1],
                z=points[:, 2],
                mode="markers",
                name="Box Targets",
                marker=dict(color="lightgray", size=4),
            )
        )

    def add_plan_trace(path, name, color, group, dash=None):
        if not path:
            return
        positions = np.array([pose[:3, 3] for pose in path])
        fig.add_trace(
            go.Scatter3d(
                x=positions[:, 0],
                y=positions[:, 1],
                z=positions[:, 2],
                mode="lines+markers",
                name=name,
                legendgroup=group,
                line=dict(color=color, width=4, dash=dash or "solid"),
                marker=dict(size=4, color=color),
            )
        )

    add_plan_trace(planned_path_dist, "Plan Dist", "crimson", "plan_dist")
    add_plan_trace(planned_path_rot, "Plan Rot", "darkorchid", "plan_rot")
    add_plan_trace(planned_path_rev, "Plan Bidir", "seagreen", "plan_bidir")
    if smooth_dist:
        add_plan_trace(smooth_dist, "Plan Dist Smooth", "tomato", "plan_dist", dash="dot")
        add_pose_axis_markers(fig, smooth_dist, legendgroup="plan_dist", scale=0.35)
    if smooth_rot:
        add_plan_trace(smooth_rot, "Plan Rot Smooth", "mediumpurple", "plan_rot", dash="dot")
        add_pose_axis_markers(fig, smooth_rot, legendgroup="plan_rot", scale=0.35)
    if smooth_rev:
        add_plan_trace(smooth_rev, "Plan Bidir Smooth", "mediumseagreen", "plan_bidir", dash="dot")
        add_pose_axis_markers(fig, smooth_rev, legendgroup="plan_bidir", scale=0.35)

    for pose in planned_path_dist:
        draw_pose_axes(fig, pose, scale=0.6, opacity=0.45, legendgroup="plan_dist")
    for pose in planned_path_rot:
        draw_pose_axes(fig, pose, scale=0.6, opacity=0.45, legendgroup="plan_rot")
    for pose in planned_path_rev:
        draw_pose_axes(fig, pose, scale=0.6, opacity=0.45, legendgroup="plan_bidir")

    ordered_points_dist, labels_dist = build_visit_annotation(points, visit_order_dist, prefix="D")
    if ordered_points_dist is not None:
        fig.add_trace(
            go.Scatter3d(
                x=ordered_points_dist[:, 0],
                y=ordered_points_dist[:, 1],
                z=ordered_points_dist[:, 2],
                mode="markers+text",
                name="Order Dist",
                legendgroup="plan_dist",
                marker=dict(color="darkorange", size=6),
                text=labels_dist,
                textposition="top center",
            )
        )

    ordered_points_rot, labels_rot = build_visit_annotation(points, visit_order_rot, prefix="R")
    if ordered_points_rot is not None:
        fig.add_trace(
            go.Scatter3d(
                x=ordered_points_rot[:, 0],
                y=ordered_points_rot[:, 1],
                z=ordered_points_rot[:, 2],
                mode="markers+text",
                name="Order Rot",
                legendgroup="plan_rot",
                marker=dict(color="royalblue", size=6),
                text=labels_rot,
                textposition="bottom center",
            )
        )

    ordered_points_rev, labels_rev = build_visit_annotation(points, visit_order_rev, prefix="B")
    if ordered_points_rev is not None:
        fig.add_trace(
            go.Scatter3d(
                x=ordered_points_rev[:, 0],
                y=ordered_points_rev[:, 1],
                z=ordered_points_rev[:, 2],
                mode="markers+text",
                name="Order Bidir",
                legendgroup="plan_bidir",
                marker=dict(color="seagreen", size=6),
                text=labels_rev,
                textposition="middle right",
            )
        )

    layout = dict(
        title=f"Box {box_idx} -> Box {box_idx+1}: Punkte {len(points)}",
        scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode="data"),
        margin=dict(l=0, r=0, t=40, b=0),
    )
    if zoom_box:
        corners = np.asarray(current_box["corners"])
        mins = corners.min(axis=0) - 2.0
        maxs = corners.max(axis=0) + 2.0
        layout["scene"].update(
            xaxis=dict(range=[mins[0], maxs[0]]),
            yaxis=dict(range=[mins[1], maxs[1]]),
            zaxis=dict(range=[mins[2], maxs[2]]),
        )
    fig.update_layout(**layout)
    return fig, planned_path_dist, planned_path_rot, planned_path_rev, pose_idx, next_pose_idx, visit_order_dist, visit_order_rot, visit_order_rev, smooth_dist, smooth_rot, smooth_rev


In [52]:
def compute_full_plan(planner):
    aggregated_path = []
    visit_orders = []
    index_pairs = []
    for box_idx in range(len(boxes) - 1):
        current_box = boxes[box_idx]
        next_box = boxes[box_idx + 1]
        pose_idx, start_pose, start_rot = first_pose_index_in_box(poses, current_box)
        next_pose_idx, goal_pose, goal_rot = first_pose_index_in_box(poses, next_box)
        if pose_idx is None or next_pose_idx is None:
            continue
        points = filter_points_in_box(target_points, current_box)
        planned_path, visit_order = planner(start_pose, start_rot, points, goal_pose, goal_rot)
        if not planned_path:
            continue
        if not aggregated_path:
            aggregated_path.extend(planned_path)
        else:
            aggregated_path.extend(planned_path[1:])
        visit_orders.append((box_idx, visit_order))
        index_pairs.append((pose_idx, next_pose_idx))
    return aggregated_path, visit_orders, index_pairs


def show_full_trajectory():
    full_dist, orders_dist, idx_dist = compute_full_plan(plan_path_through_points_distance)
    full_rot, orders_rot, idx_rot = compute_full_plan(plan_path_through_points_rotation)
    if not full_dist and not full_rot:
        raise ValueError("Keine vollständige Trajektorie verfügbar")

    fig = go.Figure()
    fig.add_trace(
        go.Scatter3d(
            x=poses[:, 0],
            y=poses[:, 1],
            z=poses[:, 2],
            mode="lines",
            name="Trajektorie",
            line=dict(color="steelblue", width=2),
            opacity=0.2,
        )
    )
    if len(target_points):
        fig.add_trace(
            go.Scatter3d(
                x=target_points[:, 0],
                y=target_points[:, 1],
                z=target_points[:, 2],
                mode="markers",
                name="Alle TargetPoints",
                marker=dict(color="lightgray", size=3),
                opacity=0.5,
            )
        )
    if full_dist:
        path_positions = np.array([pose[:3, 3] for pose in full_dist])
        #fig.add_trace(
        #    go.Scatter3d(
        #        x=path_positions[:, 0],
        #        y=path_positions[:, 1],
        #        z=path_positions[:, 2],
        #        mode="lines+markers",
        #        name="Full Dist",
        #        legendgroup="plan_dist",
        #        line=dict(color="crimson", width=4),
        #        marker=dict(size=3, color="crimson"),
        #    )
        #)
        for pose in full_dist:
            draw_pose_axes(fig, pose, scale=0.5, opacity=0.35, legendgroup="plan_dist")
    if full_rot:
        path_positions = np.array([pose[:3, 3] for pose in full_rot])
        #fig.add_trace(
        #    go.Scatter3d(
        #        x=path_positions[:, 0],
        #        y=path_positions[:, 1],
        #        z=path_positions[:, 2],
        #        mode="lines+markers",
        #        name="Full Rot",
        #        legendgroup="plan_rot",
        #        line=dict(color="darkorchid", width=4),
        #        marker=dict(size=3, color="darkorchid"),
        #    )
        #)
        for pose in full_rot:
            draw_pose_axes(fig, pose, scale=0.5, opacity=0.35, legendgroup="plan_rot")
    fig.update_layout(
        title="Komplette geplante Trajektorie (Vergleich)",
        scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode="data"),
        margin=dict(l=0, r=0, t=40, b=0),
    )
    return fig, (orders_dist, idx_dist), (orders_rot, idx_rot)


In [ ]:
from pathlib import Path

box_options = [(f"Box {i} -> {i+1}", i) for i in range(len(boxes) - 1)]
box_dropdown = widgets.Dropdown(options=box_options, description="Box", value=0)
smooth_toggle = widgets.Checkbox(value=True, description="Smooth Paths")
zoom_button = widgets.Button(description="Zoom to Box", button_style="info")
reset_button = widgets.Button(description="Reset Zoom")
full_button = widgets.Button(description="Show Full Trajectory", button_style="success")
figure_output = widgets.Output()
info_output = widgets.Output()
full_output = widgets.Output()
export_output = widgets.Output()
status = dict(last_zoom=False)
plan_cache = {}
export_dir = Path("exports")
export_dir.mkdir(exist_ok=True)
export_variant = widgets.Dropdown(
    options=[("Distance", "dist"), ("Rotation", "rot"), ("Bidirectional", "bidir")],
    description="Plan",
    value="dist",
)
export_use_smooth = widgets.Checkbox(value=True, description="Smooth")
export_frame = widgets.Text(value="odom", description="Frame")
export_filename = widgets.Text(value="plan_export.json", description="File")
export_button = widgets.Button(description="Export Poses", button_style="warning")


def poses_to_payload(poses, frame_id):
    payload = []
    for i, pose in enumerate(poses):
        position = pose[:3, 3]
        quat = R.from_matrix(pose[:3, :3]).as_quat()
        payload.append(
            {
                "index": i,
                "frame_id": frame_id,
                "position": {"x": float(position[0]), "y": float(position[1]), "z": float(position[2])},
                "orientation": {
                    "x": float(quat[0]),
                    "y": float(quat[1]),
                    "z": float(quat[2]),
                    "w": float(quat[3]),
                },
            }
        )
    return payload


def export_plan_callback(button):
    with export_output:
        export_output.clear_output(wait=True)
        if not plan_cache:
            print("Keine Plan-Daten vorhanden. Bitte zuerst rendern.")
            return
        variant = export_variant.value
        use_smooth = export_use_smooth.value
        paths = plan_cache.get("smooth" if use_smooth else "paths", {})
        poses = paths.get(variant)
        if not poses:
            print("Keine Posen für die Auswahl vorhanden.")
            return
        frame = export_frame.value.strip() or "odom"
        payload = {
            "box_index": plan_cache.get("box_idx"),
            "variant": variant,
            "smooth": use_smooth,
            "frame_id": frame,
            "poses": poses_to_payload(poses, frame),
        }
        target = (export_dir / export_filename.value).resolve()
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(json.dumps(payload, indent=2))
        print(f"Exportiert {len(poses)} Posen nach {target}")


def render_plan(zoom_box=None):
    global plan_cache
    if zoom_box is None:
        zoom_box = status['last_zoom']
    status['last_zoom'] = zoom_box
    with figure_output:
        figure_output.clear_output(wait=True)
        (
            fig,
            path_dist,
            path_rot,
            path_rev,
            start_idx,
            goal_idx,
            order_dist,
            order_rot,
            order_rev,
            smooth_dist,
            smooth_rot,
            smooth_rev,
        ) = visualize_plan(
            box_dropdown.value,
            zoom_box=zoom_box,
            smooth_enabled=smooth_toggle.value,
        )
        fig.show()
    plan_cache = {
        "box_idx": box_dropdown.value,
        "paths": {"dist": path_dist, "rot": path_rot, "bidir": path_rev},
        "smooth": {"dist": smooth_dist, "rot": smooth_rot, "bidir": smooth_rev},
    }
    export_filename.value = f"box_{box_dropdown.value}_{export_variant.value}_{'smooth' if export_use_smooth.value else 'raw'}.json"
    with info_output:
        info_output.clear_output(wait=True)
        print(f"Start Pose Index: {start_idx}")
        print(f"Ziel Pose Index: {goal_idx}")
        print(f"Pfadlänge Dist (Posen): {len(path_dist)}")
        print(f"Pfadlänge Dist Smooth (Posen): {len(smooth_dist)}")
        print(f"Pfadlänge Rot (Posen): {len(path_rot)}")
        print(f"Pfadlänge Rot Smooth (Posen): {len(smooth_rot)}")
        print(f"Pfadlänge Bidir (Posen): {len(path_rev)}")
        print(f"Pfadlänge Bidir Smooth (Posen): {len(smooth_rev)}")
        print(f"Zoom aktiv: {zoom_box}")
        print(f"Besuchsreihenfolge Dist: {order_dist}")
        print(f"Besuchsreihenfolge Rot: {order_rot}")
        print(f"Besuchsreihenfolge Bidir: {order_rev}")


def on_box_change(change):
    render_plan(zoom_box=False)


def on_zoom_clicked(button):
    render_plan(zoom_box=True)


def on_reset_clicked(button):
    render_plan(zoom_box=False)


def on_full_clicked(button):
    with full_output:
        full_output.clear_output(wait=True)
        try:
            fig, dist_info, rot_info = show_full_trajectory()
            fig.show()
            orders_dist, idx_dist = dist_info
            orders_rot, idx_rot = rot_info
            print("Distanz-basiert:")
            for (box_idx, order), (start_idx, goal_idx) in zip(orders_dist, idx_dist):
                print(f"  Box {box_idx}: Start {start_idx} -> Ziel {goal_idx}, Reihenfolge {order}")
            print("Rotation-bewusst:")
            for (box_idx, order), (start_idx, goal_idx) in zip(orders_rot, idx_rot):
                print(f"  Box {box_idx}: Start {start_idx} -> Ziel {goal_idx}, Reihenfolge {order}")
        except ValueError as exc:
            print(exc)


box_dropdown.observe(on_box_change, names="value")
zoom_button.on_click(on_zoom_clicked)
reset_button.on_click(on_reset_clicked)
full_button.on_click(on_full_clicked)
export_button.on_click(export_plan_callback)
display(widgets.HBox([box_dropdown, smooth_toggle, zoom_button, reset_button, full_button]))
display(widgets.HBox([export_variant, export_use_smooth, export_frame]))
display(widgets.HBox([export_filename, export_button]))
display(export_output)
display(figure_output)
display(info_output)
display(full_output)
render_plan(zoom_box=False)


Output()

Output()

Output()

Output()